In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Raiz do projeto. A logica abaixo permite executar o notebook tanto a
# partir da raiz quanto dentro da pasta notebooks.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.metadata_caf import METADATA_CAF, UF_NOME_POR_CD, UF_SIGLA_POR_CD

ANALYSIS_READY_PATH = ROOT / "dados" / "dados_processados" / "outputs_pu" / "trilhaB_b2_scores_all_analysis_ready.parquet"

df = pd.read_parquet(ANALYSIS_READY_PATH)
print(df.head())


In [ ]:
df


# Alta propensão estimada sem acesso observado: barreiras de acesso e casos prioritários para busca ativa

## Comparação por estado

Nesta seção comparamos, por UF, o universo total de agricultores do dataframe analítico, o subconjunto `S=1` e os recortes operacionais `flag_top100k_raw` e `flag_bottom100k_raw`.
A tabela final exibe o código da UF, a sigla e o nome por extenso usando o mapping centralizado em `src.metadata_caf`.
Os indicadores de `top100k` e `bottom100k` são interpretados como rankings operacionais dentro do universo `S=0`, e não como acesso observado.
Na versão atual da base, `340` registros sem `cd_uf` são excluídos da análise por estado. Por isso, os percentuais dos rankings por UF consideram apenas agricultores com UF conhecida.


In [ ]:
# ---------------------------------------------------------
# Tabela comparativa por UF: total, S=1 e rankings operacionais
# ---------------------------------------------------------
#
# Objetivo:
# Comparar a distribuicao estadual do universo total de agricultores,
# dos positivos observados (S=1) e dos recortes operacionais top100k e
# bottom100k produzidos pelo scoring da Trilha B.
#
# Interpretacao:
# - percent_total e percent_s1 medem a participacao estadual nos dois
#   universos principais da analise.
# - percent_top100k_nacional e percent_bottom100k_nacional medem quanto
#   cada UF participa dos conjuntos ranqueados com UF conhecida.
# - rate_top100k_no_estado e rate_bottom100k_no_estado medem a intensidade
#   local desses recortes dentro do universo S=0 da propria UF.

# Registros sem cd_uf nao podem ser distribuidos espacialmente e, por isso,
# ficam fora da tabela analitica por estado.
missing_cd_uf = int(df["cd_uf"].isna().sum())
missing_top100k_cd_uf = int(df.loc[df["flag_top100k_raw"] == 1, "cd_uf"].isna().sum())
missing_bottom100k_cd_uf = int(df.loc[df["flag_bottom100k_raw"] == 1, "cd_uf"].isna().sum())

df_analise_uf = df.loc[df["cd_uf"].notna()].copy()

df_total_uf = (
    df_analise_uf.groupby("cd_uf")["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_total")
)

df_s1_uf = (
    df_analise_uf.loc[df_analise_uf["s"] == 1]
    .groupby("cd_uf")["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_s1")
)

df_s0_uf = (
    df_analise_uf.loc[df_analise_uf["s"] == 0]
    .groupby("cd_uf")["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_s0")
)

df_top100k_uf = (
    df_analise_uf.loc[(df_analise_uf["s"] == 0) & (df_analise_uf["flag_top100k_raw"] == 1)]
    .groupby("cd_uf")["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_top100k")
)

df_bottom100k_uf = (
    df_analise_uf.loc[(df_analise_uf["s"] == 0) & (df_analise_uf["flag_bottom100k_raw"] == 1)]
    .groupby("cd_uf")["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_bottom100k")
)

df_uf_resumo = (
    df_total_uf
    .merge(df_s1_uf, on="cd_uf", how="left", validate="1:1")
    .merge(df_s0_uf, on="cd_uf", how="left", validate="1:1")
    .merge(df_top100k_uf, on="cd_uf", how="left", validate="1:1")
    .merge(df_bottom100k_uf, on="cd_uf", how="left", validate="1:1")
)

for col in ["count_s1", "count_s0", "count_top100k", "count_bottom100k"]:
    df_uf_resumo[col] = df_uf_resumo[col].fillna(0).astype("int64")

total_agricultores = df_uf_resumo["count_total"].sum()
total_s1 = df_uf_resumo["count_s1"].sum()
total_s0 = df_uf_resumo["count_s0"].sum()
total_top100k = df_uf_resumo["count_top100k"].sum()
total_bottom100k = df_uf_resumo["count_bottom100k"].sum()

df_uf_resumo["cd_uf"] = pd.to_numeric(df_uf_resumo["cd_uf"], errors="coerce").astype("Int64")
df_uf_resumo["uf_sigla"] = df_uf_resumo["cd_uf"].map(UF_SIGLA_POR_CD)
df_uf_resumo["uf_nome"] = df_uf_resumo["cd_uf"].map(UF_NOME_POR_CD)

df_uf_resumo["percent_total"] = (df_uf_resumo["count_total"] / total_agricultores * 100).round(2)

if total_s1 > 0:
    df_uf_resumo["percent_s1"] = (df_uf_resumo["count_s1"] / total_s1 * 100).round(2)
else:
    df_uf_resumo["percent_s1"] = 0.0

if total_top100k > 0:
    df_uf_resumo["percent_top100k_nacional"] = (df_uf_resumo["count_top100k"] / total_top100k * 100).round(2)
else:
    df_uf_resumo["percent_top100k_nacional"] = 0.0

if total_bottom100k > 0:
    df_uf_resumo["percent_bottom100k_nacional"] = (df_uf_resumo["count_bottom100k"] / total_bottom100k * 100).round(2)
else:
    df_uf_resumo["percent_bottom100k_nacional"] = 0.0

df_uf_resumo["rate_top100k_no_estado"] = 0.0
df_uf_resumo.loc[df_uf_resumo["count_s0"] > 0, "rate_top100k_no_estado"] = (
    df_uf_resumo.loc[df_uf_resumo["count_s0"] > 0, "count_top100k"]
    / df_uf_resumo.loc[df_uf_resumo["count_s0"] > 0, "count_s0"]
    * 100
).round(2)

df_uf_resumo["rate_bottom100k_no_estado"] = 0.0
df_uf_resumo.loc[df_uf_resumo["count_s0"] > 0, "rate_bottom100k_no_estado"] = (
    df_uf_resumo.loc[df_uf_resumo["count_s0"] > 0, "count_bottom100k"]
    / df_uf_resumo.loc[df_uf_resumo["count_s0"] > 0, "count_s0"]
    * 100
).round(2)

ordered_cols = [
    "cd_uf",
    "uf_sigla",
    "uf_nome",
    "count_total",
    "percent_total",
    "count_s1",
    "percent_s1",
    "count_s0",
    "count_top100k",
    "percent_top100k_nacional",
    "rate_top100k_no_estado",
    "count_bottom100k",
    "percent_bottom100k_nacional",
    "rate_bottom100k_no_estado",
]

df_uf_resumo = (
    df_uf_resumo[ordered_cols]
    .sort_values(["count_top100k", "count_s1", "count_total"], ascending=[False, False, False])
    .reset_index(drop=True)
)

df_uf_resumo


In [ ]:
# Totais agregados usados na leitura da tabela por UF.
print(f"Registros excluidos da analise por cd_uf ausente: {missing_cd_uf:,}")
print(f"Casos top100k excluidos por cd_uf ausente: {missing_top100k_cd_uf:,}")
print(f"Casos bottom100k excluidos por cd_uf ausente: {missing_bottom100k_cd_uf:,}")
print(f"Total de agricultores com UF conhecida: {total_agricultores:,}")
print(f"Total de agricultores com s=1 e UF conhecida: {total_s1:,}")
print(f"Total de agricultores com s=0 e UF conhecida: {total_s0:,}")
print(f"Total de agricultores no top100k com UF conhecida: {total_top100k:,}")
print(f"Total de agricultores no bottom100k com UF conhecida: {total_bottom100k:,}")


## Comparação por papel do membro familiar

Nesta seção avaliamos a distribuição de `id_tipo_membro_familiar` no universo total, em `S=1`, em `S=0` e nos recortes operacionais `top100k` e `bottom100k`.
A leitura é descritiva: o papel do agricultor na família ajuda a caracterizar quem concentra acesso observado e quem aparece com maior ou menor prioridade no ranking operacional.
Como alguns grupos são pequenos, as taxas devem ser interpretadas sempre junto com as contagens absolutas.


In [ ]:
# ---------------------------------------------------------
# Tabela comparativa por papel do membro familiar
# ---------------------------------------------------------

membro_map = METADATA_CAF["id_tipo_membro_familiar"]

df["tipo_membro_familiar"] = (
    pd.to_numeric(df["id_tipo_membro_familiar"], errors="coerce")
    .astype("Int64")
    .map(membro_map)
    .fillna("Nao mapeado")
)

df_total_membro = (
    df.groupby(["id_tipo_membro_familiar", "tipo_membro_familiar"], dropna=False)["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_total")
)

df_s1_membro = (
    df.loc[df["s"] == 1]
    .groupby(["id_tipo_membro_familiar", "tipo_membro_familiar"], dropna=False)["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_s1")
)

df_s0_membro = (
    df.loc[df["s"] == 0]
    .groupby(["id_tipo_membro_familiar", "tipo_membro_familiar"], dropna=False)["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_s0")
)

df_top100k_membro = (
    df.loc[(df["s"] == 0) & (df["flag_top100k_raw"] == 1)]
    .groupby(["id_tipo_membro_familiar", "tipo_membro_familiar"], dropna=False)["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_top100k")
)

df_bottom100k_membro = (
    df.loc[(df["s"] == 0) & (df["flag_bottom100k_raw"] == 1)]
    .groupby(["id_tipo_membro_familiar", "tipo_membro_familiar"], dropna=False)["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_bottom100k")
)

df_membro_resumo = (
    df_total_membro
    .merge(df_s1_membro, on=["id_tipo_membro_familiar", "tipo_membro_familiar"], how="left", validate="1:1")
    .merge(df_s0_membro, on=["id_tipo_membro_familiar", "tipo_membro_familiar"], how="left", validate="1:1")
    .merge(df_top100k_membro, on=["id_tipo_membro_familiar", "tipo_membro_familiar"], how="left", validate="1:1")
    .merge(df_bottom100k_membro, on=["id_tipo_membro_familiar", "tipo_membro_familiar"], how="left", validate="1:1")
)

for col in ["count_s1", "count_s0", "count_top100k", "count_bottom100k"]:
    df_membro_resumo[col] = df_membro_resumo[col].fillna(0).astype("int64")

total_membro = df_membro_resumo["count_total"].sum()
total_s1_membro = df_membro_resumo["count_s1"].sum()
total_s0_membro = df_membro_resumo["count_s0"].sum()
total_top100k_membro = df_membro_resumo["count_top100k"].sum()
total_bottom100k_membro = df_membro_resumo["count_bottom100k"].sum()

df_membro_resumo["percent_total"] = (df_membro_resumo["count_total"] / total_membro * 100).round(2)

if total_s1_membro > 0:
    df_membro_resumo["percent_s1_nacional"] = (df_membro_resumo["count_s1"] / total_s1_membro * 100).round(2)
else:
    df_membro_resumo["percent_s1_nacional"] = 0.0

df_membro_resumo["rate_s1_no_grupo"] = (
    df_membro_resumo["count_s1"] / df_membro_resumo["count_total"] * 100
).round(2)

if total_top100k_membro > 0:
    df_membro_resumo["percent_top100k_nacional"] = (df_membro_resumo["count_top100k"] / total_top100k_membro * 100).round(2)
else:
    df_membro_resumo["percent_top100k_nacional"] = 0.0

df_membro_resumo["rate_top100k_no_s0"] = 0.0
df_membro_resumo.loc[df_membro_resumo["count_s0"] > 0, "rate_top100k_no_s0"] = (
    df_membro_resumo.loc[df_membro_resumo["count_s0"] > 0, "count_top100k"]
    / df_membro_resumo.loc[df_membro_resumo["count_s0"] > 0, "count_s0"]
    * 100
).round(2)

if total_bottom100k_membro > 0:
    df_membro_resumo["percent_bottom100k_nacional"] = (df_membro_resumo["count_bottom100k"] / total_bottom100k_membro * 100).round(2)
else:
    df_membro_resumo["percent_bottom100k_nacional"] = 0.0

df_membro_resumo["rate_bottom100k_no_s0"] = 0.0
df_membro_resumo.loc[df_membro_resumo["count_s0"] > 0, "rate_bottom100k_no_s0"] = (
    df_membro_resumo.loc[df_membro_resumo["count_s0"] > 0, "count_bottom100k"]
    / df_membro_resumo.loc[df_membro_resumo["count_s0"] > 0, "count_s0"]
    * 100
).round(2)

ordered_cols = [
    "id_tipo_membro_familiar",
    "tipo_membro_familiar",
    "count_total",
    "percent_total",
    "count_s1",
    "percent_s1_nacional",
    "rate_s1_no_grupo",
    "count_s0",
    "count_top100k",
    "percent_top100k_nacional",
    "rate_top100k_no_s0",
    "count_bottom100k",
    "percent_bottom100k_nacional",
    "rate_bottom100k_no_s0",
]

df_membro_resumo = (
    df_membro_resumo[ordered_cols]
    .sort_values(["id_tipo_membro_familiar", "tipo_membro_familiar"], ascending=[True, True])
    .reset_index(drop=True)
)

df_membro_resumo


In [ ]:
# Grupos pequenos na leitura detalhada por papel familiar.
grupos_pequenos = df_membro_resumo.loc[
    df_membro_resumo["count_total"] < 1000,
    ["id_tipo_membro_familiar", "tipo_membro_familiar", "count_total"],
]

print(f"Total de agricultores no dataset: {total_membro:,}")
print(f"Total de agricultores com s=1: {total_s1_membro:,}")
print(f"Total de agricultores com s=0: {total_s0_membro:,}")
print(f"Total de agricultores no top100k: {total_top100k_membro:,}")
print(f"Total de agricultores no bottom100k: {total_bottom100k_membro:,}")
print(f"Quantidade de grupos com menos de 1.000 agricultores: {len(grupos_pequenos):,}")

grupos_pequenos


## Tabela agrupada a partir de `df_membro_resumo`

O chunk abaixo parte da tabela detalhada `df_membro_resumo` e reúne em `Outros (<1%)` as categorias cuja `percent_total` é menor que `1`.
Depois do agrupamento, as métricas percentuais e as taxas são recalculadas a partir das contagens somadas para manter consistência analítica.


In [ ]:
# ---------------------------------------------------------
# df_membro_resumo com categorias de baixa representacao agrupadas
# ---------------------------------------------------------

threshold_percent_total = 1.0

df_membro_resumo_agrupado = df_membro_resumo.copy()
df_membro_resumo_agrupado["grupo_membro_familiar"] = df_membro_resumo_agrupado["tipo_membro_familiar"].where(
    df_membro_resumo_agrupado["percent_total"] >= threshold_percent_total,
    "Outros (<1%)",
)

ordem_grupo_membro = df_membro_resumo_agrupado.loc[
    df_membro_resumo_agrupado["percent_total"] >= threshold_percent_total,
    "tipo_membro_familiar",
].tolist()
if (df_membro_resumo_agrupado["grupo_membro_familiar"] == "Outros (<1%)").any():
    ordem_grupo_membro.append("Outros (<1%)")

cols_count = ["count_total", "count_s1", "count_s0", "count_top100k", "count_bottom100k"]

df_membro_resumo_agrupado = (
    df_membro_resumo_agrupado
    .groupby("grupo_membro_familiar", dropna=False)[cols_count]
    .sum()
    .reset_index()
)

df_membro_resumo_agrupado["percent_total"] = (
    df_membro_resumo_agrupado["count_total"] / total_membro * 100
).round(2)

if total_s1_membro > 0:
    df_membro_resumo_agrupado["percent_s1_nacional"] = (
        df_membro_resumo_agrupado["count_s1"] / total_s1_membro * 100
    ).round(2)
else:
    df_membro_resumo_agrupado["percent_s1_nacional"] = 0.0

df_membro_resumo_agrupado["rate_s1_no_grupo"] = (
    df_membro_resumo_agrupado["count_s1"] / df_membro_resumo_agrupado["count_total"] * 100
).round(2)

if total_top100k_membro > 0:
    df_membro_resumo_agrupado["percent_top100k_nacional"] = (
        df_membro_resumo_agrupado["count_top100k"] / total_top100k_membro * 100
    ).round(2)
else:
    df_membro_resumo_agrupado["percent_top100k_nacional"] = 0.0

df_membro_resumo_agrupado["rate_top100k_no_s0"] = 0.0
df_membro_resumo_agrupado.loc[df_membro_resumo_agrupado["count_s0"] > 0, "rate_top100k_no_s0"] = (
    df_membro_resumo_agrupado.loc[df_membro_resumo_agrupado["count_s0"] > 0, "count_top100k"]
    / df_membro_resumo_agrupado.loc[df_membro_resumo_agrupado["count_s0"] > 0, "count_s0"]
    * 100
).round(2)

if total_bottom100k_membro > 0:
    df_membro_resumo_agrupado["percent_bottom100k_nacional"] = (
        df_membro_resumo_agrupado["count_bottom100k"] / total_bottom100k_membro * 100
    ).round(2)
else:
    df_membro_resumo_agrupado["percent_bottom100k_nacional"] = 0.0

df_membro_resumo_agrupado["rate_bottom100k_no_s0"] = 0.0
df_membro_resumo_agrupado.loc[df_membro_resumo_agrupado["count_s0"] > 0, "rate_bottom100k_no_s0"] = (
    df_membro_resumo_agrupado.loc[df_membro_resumo_agrupado["count_s0"] > 0, "count_bottom100k"]
    / df_membro_resumo_agrupado.loc[df_membro_resumo_agrupado["count_s0"] > 0, "count_s0"]
    * 100
).round(2)

ordered_cols_agrupado = [
    "grupo_membro_familiar",
    "count_total",
    "percent_total",
    "count_s1",
    "percent_s1_nacional",
    "rate_s1_no_grupo",
    "count_s0",
    "count_top100k",
    "percent_top100k_nacional",
    "rate_top100k_no_s0",
    "count_bottom100k",
    "percent_bottom100k_nacional",
    "rate_bottom100k_no_s0",
]

df_membro_resumo_agrupado = df_membro_resumo_agrupado[ordered_cols_agrupado]
df_membro_resumo_agrupado["grupo_membro_familiar"] = pd.Categorical(
    df_membro_resumo_agrupado["grupo_membro_familiar"],
    categories=ordem_grupo_membro,
    ordered=True,
)
df_membro_resumo_agrupado = (
    df_membro_resumo_agrupado
    .sort_values("grupo_membro_familiar")
    .reset_index(drop=True)
)

df_membro_resumo_agrupado


In [ ]:
# Categorias detalhadas que foram reunidas em Outros (<1%).
categorias_agrupadas_em_outros = (
    df_membro_resumo.loc[
        df_membro_resumo["percent_total"] < threshold_percent_total,
        ["id_tipo_membro_familiar", "tipo_membro_familiar", "count_total", "percent_total"],
    ]
    .sort_values(["count_total", "id_tipo_membro_familiar"], ascending=[False, True])
    .reset_index(drop=True)
)

categorias_agrupadas_em_outros


## Comparação conjunta por grupo do membro familiar e sexo

Nesta seção cruzamos o agrupamento de `grupo_membro` com o sexo do agricultor para verificar como a composição demográfica se relaciona com `S=1` e com os recortes operacionais `top100k` e `bottom100k`.
A tabela exibe, para cada combinação `grupo_membro x sexo`, o total absoluto de agricultores na base (`S=0` e `S=1`) e o total absoluto de agricultores com acesso observado ao crédito (`S=1`).
Cada percentual usa o total do universo filtrado correspondente como denominador. Assim, `percent_total` usa a base inteira, `rate_s1` usa o total de agricultores com `S=1`, `percent_top100k` usa o total com `S=0` e `flag_top100k_raw = 1`, e `percent_bottom100k` usa o total com `S=0` e `flag_bottom100k_raw = 1`.
Essa leitura mostra a composição relativa de cada combinação dentro de cada universo analítico.


In [ ]:
# ---------------------------------------------------------
# Tabela comparativa por grupo do membro familiar e sexo
# ---------------------------------------------------------
#
# Objetivo:
# Cruzar o agrupamento por papel do membro familiar com o sexo para
# identificar quais combinacoes concentram maior presenca no total da
# base e qual a participacao relativa de cada combinacao nos universos
# S=1, top100k e bottom100k.
#
# Interpretacao:
# - total_agricultores e total_agricultores_s1 sao contagens absolutas
#   da propria combinacao grupo_membro x sexo.
# - percent_total usa como denominador o total de agricultores da base.
# - rate_s1 usa como denominador o total de agricultores com S=1.
# - percent_top100k usa como denominador o total de agricultores com
#   S=0 e flag_top100k_raw=1.
# - percent_bottom100k usa como denominador o total de agricultores com
#   S=0 e flag_bottom100k_raw=1.

sexo_map = METADATA_CAF["id_sexo"]

grupo_membro_map = (
    df_membro_resumo.assign(
        grupo_membro=df_membro_resumo["tipo_membro_familiar"].where(
            df_membro_resumo["percent_total"] >= threshold_percent_total,
            "Outros (<1%)",
        )
    )
    .set_index("id_tipo_membro_familiar")["grupo_membro"]
    .to_dict()
)

df["grupo_membro"] = (
    pd.to_numeric(df["id_tipo_membro_familiar"], errors="coerce")
    .astype("Int64")
    .map(grupo_membro_map)
    .fillna("Nao mapeado")
)

df["sexo"] = (
    pd.to_numeric(df["id_sexo"], errors="coerce")
    .astype("Int64")
    .map(sexo_map)
    .fillna("Nao mapeado")
)

total_agricultores_base = df["cpf_agricultor"].nunique()
total_agricultores_s1 = df.loc[df["s"] == 1, "cpf_agricultor"].nunique()
total_agricultores_top100k = df.loc[(df["s"] == 0) & (df["flag_top100k_raw"] == 1), "cpf_agricultor"].nunique()
total_agricultores_bottom100k = df.loc[(df["s"] == 0) & (df["flag_bottom100k_raw"] == 1), "cpf_agricultor"].nunique()

df_total_membro_sexo = (
    df.groupby(["grupo_membro", "sexo"], dropna=False)["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_total")
)

df_s1_membro_sexo = (
    df.loc[df["s"] == 1]
    .groupby(["grupo_membro", "sexo"], dropna=False)["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_s1")
)

df_s0_membro_sexo = (
    df.loc[df["s"] == 0]
    .groupby(["grupo_membro", "sexo"], dropna=False)["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_s0")
)

df_top100k_membro_sexo = (
    df.loc[(df["s"] == 0) & (df["flag_top100k_raw"] == 1)]
    .groupby(["grupo_membro", "sexo"], dropna=False)["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_top100k")
)

df_bottom100k_membro_sexo = (
    df.loc[(df["s"] == 0) & (df["flag_bottom100k_raw"] == 1)]
    .groupby(["grupo_membro", "sexo"], dropna=False)["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_bottom100k")
)

df_membro_sexo_tecnico = (
    df_total_membro_sexo
    .merge(df_s1_membro_sexo, on=["grupo_membro", "sexo"], how="left", validate="1:1")
    .merge(df_s0_membro_sexo, on=["grupo_membro", "sexo"], how="left", validate="1:1")
    .merge(df_top100k_membro_sexo, on=["grupo_membro", "sexo"], how="left", validate="1:1")
    .merge(df_bottom100k_membro_sexo, on=["grupo_membro", "sexo"], how="left", validate="1:1")
)

for col in ["count_s1", "count_s0", "count_top100k", "count_bottom100k"]:
    df_membro_sexo_tecnico[col] = df_membro_sexo_tecnico[col].fillna(0).astype("int64")

df_membro_sexo_tecnico["total_agricultores"] = df_membro_sexo_tecnico["count_total"]
df_membro_sexo_tecnico["total_agricultores_s1"] = df_membro_sexo_tecnico["count_s1"]

df_membro_sexo_tecnico["percent_total"] = (
    df_membro_sexo_tecnico["count_total"] / total_agricultores_base * 100
).round(2)

if total_agricultores_s1 > 0:
    df_membro_sexo_tecnico["rate_s1"] = (
        df_membro_sexo_tecnico["count_s1"] / total_agricultores_s1 * 100
    ).round(2)
else:
    df_membro_sexo_tecnico["rate_s1"] = 0.0

if total_agricultores_top100k > 0:
    df_membro_sexo_tecnico["percent_top100k"] = (
        df_membro_sexo_tecnico["count_top100k"] / total_agricultores_top100k * 100
    ).round(2)
else:
    df_membro_sexo_tecnico["percent_top100k"] = 0.0

if total_agricultores_bottom100k > 0:
    df_membro_sexo_tecnico["percent_bottom100k"] = (
        df_membro_sexo_tecnico["count_bottom100k"] / total_agricultores_bottom100k * 100
    ).round(2)
else:
    df_membro_sexo_tecnico["percent_bottom100k"] = 0.0

ordem_sexo = [
    sexo
    for sexo in ["Masculino", "Feminino", "Outros", "Nao mapeado"]
    if sexo in df_membro_sexo_tecnico["sexo"].astype(str).unique().tolist()
]

df_membro_sexo_tecnico["grupo_membro"] = pd.Categorical(
    df_membro_sexo_tecnico["grupo_membro"],
    categories=ordem_grupo_membro,
    ordered=True,
)
df_membro_sexo_tecnico["sexo"] = pd.Categorical(
    df_membro_sexo_tecnico["sexo"],
    categories=ordem_sexo,
    ordered=True,
)

df_membro_sexo_tecnico = (
    df_membro_sexo_tecnico
    .sort_values(["grupo_membro", "sexo"])
    .reset_index(drop=True)
)

combos_muito_pequenos = df_membro_sexo_tecnico.loc[
    df_membro_sexo_tecnico["count_total"] < 1000,
    ["grupo_membro", "sexo", "count_total", "count_s0"],
]
print(f"Total de agricultores na base: {total_agricultores_base:,}")
print(f"Total de agricultores em S=1: {total_agricultores_s1:,}")
print(f"Total de agricultores em top100k: {total_agricultores_top100k:,}")
print(f"Total de agricultores em bottom100k: {total_agricultores_bottom100k:,}")
print(f"Combinacoes grupo_membro x sexo com menos de 1.000 agricultores: {len(combos_muito_pequenos):,}")

df_membro_sexo_resumo = df_membro_sexo_tecnico[
    [
        "grupo_membro",
        "sexo",
        "total_agricultores",
        "total_agricultores_s1",
        "percent_total",
        "rate_s1",
        "percent_top100k",
        "percent_bottom100k",
    ]
].copy()

df_membro_sexo_resumo


In [ ]:
# total de agricultores por sexo
df_sexo = (
    df.groupby("sexo")["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_total")
    .sort_values("count_total", ascending=False)
    .reset_index(drop=True)
)

print("Total de agricultores por sexo:")
print(df_sexo)

In [ ]:
# Total de agricultores que pegaram crédito S=1 por sexo
df_s1_sexo = (
    df.loc[df["s"] == 1]
    .groupby("sexo")["cpf_agricultor"]
    .nunique()
    .reset_index(name="count_s1")
    .sort_values("count_s1", ascending=False)
    .reset_index(drop=True)
)
print("Total de agricultores que pegaram crédito S=1 por sexo:")
print(df_s1_sexo)